# Baseline 04c — Ablación de bloques de características

Mide el aporte incremental de cada bloque del vector fused. Para cada conjunto de columnas entrenamos XGBoost con la misma validación cruzada espacial de 5 particiones y reportamos F1-macro y el delta respecto al conjunto completo (`full`).

Conjuntos canónicos evaluados:

- `full`: todas las características numéricas disponibles.
- `no_geom`: `full` sin las 3 columnas `geom_*`.
- `no_geom_no_era5_srtm`: además sin `era5_*` ni `srtm_*`.
- `alphaearth_only`: sólo las 64 dimensiones `ae_*`.
- `phenology_only`: 8 atributos fenológicos + 24 FFT NDVI.
- `geom_only`: sólo `geom_*` (prueba cuantitativa de fuga espacial).

**Detección de columnas AlphaEarth**: el detector tolera variantes de prefijo (`ae_*`, `emb_*`, `dim_*`, `alphaearth_*`), por lo que `alphaearth_only` ya no aparece con `n_features=0` ni NaN cuando hay embeddings AlphaEarth en el dataset.

In [ ]:
FEATURES_PATH = "data/test_fixtures/feature_selection_parcels_subset.parquet"
PARCELS_GEOPARQUET = "data/processed/pastis_parcels_full.geoparquet"
FIGURES_SUBDIR = "us-023-preview/04c_baseline"
REPORTS_SUBDIR = "baseline/04c_baseline"
K_FOLDS = 5
BUFFER_KM = 1.0
MAX_SAMPLES = None  # None = dataset completo; usar un valor menor para corridas rápidas.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))


## Carga del dataset y ejecución de la ablación

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from ml.utils.baseline_notebook_helpers import (
    load_features_dataset_with_meta,
    run_ablation_and_persist,
)
from ml.eval.reencuadre_plots import (
    plot_ablation_bars,
    plot_geom_leakage_comparison,
)

df = load_features_dataset_with_meta(
    path=FEATURES_PATH,
    parcels_geoparquet=PARCELS_GEOPARQUET,
)
display(Markdown(f'Dataset: `{df.height:,}` parcelas x `{df.width}` cols'))

ablation_table, parquet_path = run_ablation_and_persist(
    df,
    output_dir=env.reports_dir,
    models=('xgb',),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    max_samples=MAX_SAMPLES,
)
display(Markdown(f'**Tabla de ablación**: `{parquet_path.relative_to(env.repo)}`'))
display(ablation_table)


## Gráficos: F1-macro por conjunto y comparativa del bloque `geom_*`

In [ ]:
from ml.eval.feature_ablation import FeatureAblationResult

results = [
    FeatureAblationResult(
        feature_set=row['feature_set'],
        model_kind=row['model'],
        f1_macro=row['f1_macro'] if row['f1_macro'] is not None else float('nan'),
        f1_weighted=row['f1_weighted'] if row['f1_weighted'] is not None else float('nan'),
        miou=row['miou'] if row['miou'] is not None else float('nan'),
        n_features=row['n_features'],
        delta_vs_full=row['delta_vs_full'] if row['delta_vs_full'] is not None else float('nan'),
    )
    for row in ablation_table.iter_rows(named=True)
]

fig_abl = plot_ablation_bars(results, title='F1-macro por conjunto de características')
fig_abl.savefig(env.figures_dir / 'ablation_bars.png', bbox_inches='tight')
display(fig_abl)
plt.close(fig_abl)

fig_geom = plot_geom_leakage_comparison(results)
fig_geom.savefig(env.figures_dir / 'geom_leakage.png', bbox_inches='tight')
display(fig_geom)
plt.close(fig_geom)


## Conclusiones

**Lectura de la ablación**:

- El conjunto `full` define la referencia. El delta de `no_geom` respecto a `full` cuantifica el aporte (o ruido) de las columnas geométricas: si el delta es cercano a cero, `geom_*` no aporta señal agronómica; si es positivo, descartarlas mejora porque estaban introduciendo ruido.

- `geom_only` es la **prueba cuantitativa de fuga espacial**: si F1-macro < 0.10, confirmamos que área, perímetro y elongación por sí solas no permiten clasificar cultivos; el modelo no puede aprender la clase a partir de la geometría.

- `alphaearth_only` indica qué fracción del baseline proviene de los 64 embeddings del modelo fundacional. Si la diferencia entre `alphaearth_only` y `full` es pequeña, los demás bloques aportan poco más allá del FM.

## Lo que sigue

- `05_reencuadre_fenologico.ipynb` amplía esta tabla con los bloques opcionales (FarSLIP, descripción fenológica textual con Gemini, firma espectral REP), materializados desde el propio cuaderno si no existen en disco.
- `Avance3.Equipo17.ipynb` consume `ablation_table.parquet` para decidir el conjunto ganador.